# 5X72 fixed-P59 control: saved observations

The [prespecified workflow](5x72_reciprocal_workflow.md) compares a same-writer protein-only sham with fixed experimental P59 while searching P69. This notebook reads the original 24-cell producer archive. It executes **no docking, scoring, minimization or molecular preparation**. Recovery uses 24 experimental P69 heavy atoms, positional RMSD <=2.5 Å, no fitting or symmetry correction. Generated companion H geometry is not experimental. #17/#6/#33 remain partial/open. Fixed P59 changes any-returned P69 recovery from 0/12 to 7/12, while first-ranked and lowest frozen-score recovery stay zero.

In [1]:
import gzip, json, hashlib, sys
from pathlib import Path
checkout = next(p for p in (Path.cwd(), *Path.cwd().parents)
    if (p/'devtools/qualify_5x72_reciprocal.py').is_file())
sys.path.insert(0, str(checkout))
import numpy as np
import pandas as pd
import pyunitwizard as puw
import dockingmt as dmt
from devtools.qualify_5x72_reciprocal import OUTPUT
from devtools.qualify_5x72_occupancy import load_baseline, measure, atom_fields

payload = OUTPUT.read_bytes()
record = json.loads(gzip.decompress(payload))
assert record['schema'] == 'dockingmt.5x72_reciprocal_audit@1'
assert len(record['new_runs']) == 24
print('Original gzip SHA256:', hashlib.sha256(payload).hexdigest())
print('Preserved provider source:', record['qualified_provider_commit'])
rows = pd.DataFrame(record['comparison_rows'])
summary = rows.groupby(['receptor', 'order']).agg(
    cells=('first_recovered', 'size'),
    first_recovered=('first_recovered', 'sum'),
    returned_set_recovered=('returned_set_recovered', 'sum'),
    returned_poses=('poses', 'sum'))
display(summary)

Original gzip SHA256: 323a5cb7a826368eb4914219c3a29c2fdf440956a792f9ebf7812e2cc228083d
Preserved provider source: 5bd893c85fe8d211663b2b1f865f5f1d2c382a90


cells  first_recovered  returned_set_recovered  \
receptor order                                                         
occupied first_fixed      6                0                       3   
         native           6                0                       4   
sham     first_fixed      6                0                       0   
         native           6                0                       0   

                      returned_poses  
receptor order                        
occupied first_fixed              14  
         native                   15  
sham     first_fixed              24  
         native                   25

All new returned poses are checked against the original P69 experimental reference under a pm/fs unit policy. The fixed occupant retains P59 heavy coordinates and contributes a generated, rigid polar H. Neither score magnitudes nor these small counts establish affinity, cooperativity, stereoselectivity, recovery probability or convergence.

In [2]:
checked = 0
with puw.context(standard_units=['pm','fs','K','mole','dalton','e','kJ/mol','radians']):
    for run in record['new_runs']:
        case = record['cases']['p69']
        result = dmt.DockingResult.from_dict(run['result'])
        control = case['controls'][run['order']]
        metrics = measure(result, case, control)
        reference = np.asarray(case['reference_coordinates_angstrom'])[control['heavy_pdbqt_to_reference_atom_indices']]
        for pose, saved, metric in zip(result.poses, run['heavy_atom_metrics'], metrics, strict=True):
            coordinates = puw.get_value(pose.coordinates, to_unit='angstrom')[control['heavy_pdbqt_indices']]
            independent = np.sqrt(np.mean(np.sum((coordinates-reference)**2, axis=1)))
            assert abs(independent-saved['heavy_atom_positional_rmsd_angstrom']) < 1e-10
            assert abs(independent-metric['heavy_atom_positional_rmsd_angstrom']) < 1e-10
            assert saved['recovered'] == bool(independent <= 2.5)
            checked += 1
assert checked == 78
print('All 78 new poses verified against the unchanged 24-heavy-atom reference.')

All 78 new poses verified against the unchanged 24-heavy-atom reference.


The following checks all 156 new-pose and 98 historical serialization evaluations. It verifies captured bytes and unchanged geometry; evaluations are read, not repeated. The lowest fixed score within each declared set is a finite diagnostic with original-rank tie breaking, distinct from Vina docking scores and public automatic ranking.

In [3]:
evaluations = 0
for run in record['new_runs']:
    originals = dmt.DockingResult.from_dict(run['result']).poses
    for original, pair in zip(originals, run['fixed_scores'], strict=True):
        for name, data in pair.items():
            pose = dmt.DockingPose.from_dict(data)
            artifacts = pose.metadata['scoring_history'][-1]['backend_artifacts']
            assert dmt.verify_captured_inputs(artifacts)
            assert artifacts['receptor']['sha256'] == record['preparation']['receptors'][name]['sha256']
            np.testing.assert_allclose(puw.get_value(pose.coordinates, to_unit='angstrom'),
                puw.get_value(original.coordinates, to_unit='angstrom'), rtol=0, atol=1e-12)
            assert pose.rank == original.rank
            evaluations += 1
baseline = [r for r in load_baseline()['new_runs'] if r['ligand']=='p69']
for prior, control in zip(baseline, record['historical_serialization_scores'], strict=True):
    for original, pair in zip(dmt.DockingResult.from_dict(prior['result']).poses, control['fixed_scores'], strict=True):
        for data in pair.values():
            pose = dmt.DockingPose.from_dict(data)
            assert dmt.verify_captured_inputs(pose.metadata['scoring_history'][-1]['backend_artifacts'])
            np.testing.assert_allclose(puw.get_value(pose.coordinates, to_unit='angstrom'),
                puw.get_value(original.coordinates, to_unit='angstrom'), rtol=0, atol=1e-12)
            evaluations += 1
        a = pair['original']['metadata']['scoring_history'][-1]['scores']
        b = pair['sham']['metadata']['scoring_history'][-1]['scores']
        assert a == b
assert evaluations == 254
print('254 captured frozen evaluations verified; all historical eight-component serialization pairs agree.')
diagnostic = []
for (receptor, order), group in rows.groupby(['receptor','order']):
    diagnostic.append({'search_receptor':receptor, 'order':order,
        'sham_lowest_recovered':sum(r['sham']['recovered'] for r in group.fixed_score_lowest),
        'occupied_lowest_recovered':sum(r['occupied']['recovered'] for r in group.fixed_score_lowest)})
display(pd.DataFrame(diagnostic))

254 captured frozen evaluations verified; all historical eight-component serialization pairs agree.


,search_receptor,order,sham_lowest_recovered,occupied_lowest_recovered
0,occupied,first_fixed,0,0
1,occupied,native,0,0
2,sham,first_fixed,0,0
3,sham,native,0,0
